# 04 - LoRA Training: Red neuronal sobre LLM

Este notebook entrena un adapter LoRA (red neuronal liviana) sobre un LLM open-source
(Mistral o LLaMA) para darle capacidad de traducción Ayoreo-Español.

## Arquitectura
```
LLM Base (frozen)  +  LoRA Adapter (trainable)
  Mistral-7B           ~1-5% parámetros
  (no se modifica)     (red neuronal que aprende Ayoreo)
```

**Requisitos**: GPU con >= 16GB VRAM (o usar Google Colab T4/A100).
Si no tenés GPU, usá `python scripts/run_finetune.py --gemini-only`.

**Instalar dependencias de training**:
```bash
pip install -e '.[training]'
```

In [ ]:
# Verificar GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Apple MPS (Metal) available — will use CPU fallback for training")

In [ ]:
# Cargar y explorar datos de entrenamiento
from src.training.lora_trainer import load_training_data, format_prompt

train_data = load_training_data()
print(f"Training examples: {len(train_data)}")
print(f"\n--- Ejemplo de prompt ---")
print(format_prompt(train_data[0]))

In [ ]:
# Ver configuración de LoRA
from src.utils.config import load_config

config = load_config('training')
lora_config = config['lora']
print(f"Base model: {lora_config['base_model']}")
print(f"LoRA rank: {lora_config['rank']}")
print(f"Epochs: {lora_config['epochs']}")
print(f"Batch size: {lora_config['batch_size']}")
print(f"Learning rate: {lora_config['learning_rate']}")

In [ ]:
# Entrenar! (esto tarda — depende de GPU y tamaño del corpus)
from src.training.lora_trainer import train

# Opción 1: usar defaults de configs/training.yaml
adapter_dir = train()

# Opción 2: override manual
# adapter_dir = train(
#     base_model="meta-llama/Llama-3.1-8B",
#     epochs=5,
#     batch_size=2,
#     learning_rate=1e-4,
# )

In [ ]:
# Probar el modelo entrenado
from src.inference.lora_translator import LoRATranslator

translator = LoRATranslator()

# Ayoreo -> Español
result = translator.translate("Dupade ome yoqui ore guejna.", direction="ayo_to_es")
print(f"AYO -> ES: {result}")

# Español -> Ayoreo
result = translator.translate("Dios nos ama.", direction="es_to_ayo")
print(f"ES -> AYO: {result}")

In [ ]:
# Evaluar en test set
import json
from src.utils.config import PROJECT_ROOT
from src.training.evaluate import compute_bleu, compute_chrf

test_path = PROJECT_ROOT / "data" / "splits" / "test.jsonl"
test_data = [json.loads(l) for l in open(test_path)]

predictions = []
references = []
for entry in test_data[:50]:  # Evaluar primeros 50
    pred = translator.translate(entry["ayoreo"], direction="ayo_to_es")
    predictions.append(pred)
    references.append(entry["spanish"])

bleu = compute_bleu(predictions, references)
chrf = compute_chrf(predictions, references)
print(f"\nBLEU: {bleu:.2f}")
print(f"chrF: {chrf:.2f}")